In [1]:
%pip install mlflow

  Using cached flask-3.1.3-py3-none-any.whl.metadata (3.2 kB)
  Using cached blinker-1.9.0-py3-none-any.whl.metadata (1.6 kB)
  Using cached itsdangerous-2.2.0-py3-none-any.whl.metadata (1.9 kB)
  Using cached python_dateutil-2.9.0.post0-py2.py3-none-any.whl.metadata (8.4 kB)
   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
    --------------------------------------- 0.3/12.6 MB ? eta -:--:--
   -- ------------------------------------- 0.8/12.6 MB 1.9 MB/s eta 0:00:07
   ---- ----------------------------------- 1.3/12.6 MB 2.2 MB/s eta 0:00:06
   ----- ---------------------------------- 1.8/12.6 MB 2.2 MB/s eta 0:00:05
   -------- ------------------------------- 2.6/12.6 MB 2.5 MB/s eta 0:00:04
   ---------- ----------------------------- 3.4/12.6 MB 2.8 MB/s eta 0:00:04
   ------------- -------------------------- 4.2/12.6 MB 3.0 MB/s eta 0:00:03
   -------------- ------------------------- 4.7/12.6 MB 2.9 MB/s eta 0:00:03
   ---------------- ----------------------- 


[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import json
import mlflow
import pandas as pd
from xgboost import XGBClassifier
from sklearn.metrics import average_precision_score, precision_score, recall_score, f1_score

# Store experiment data in a database file in the project root
mlflow.set_tracking_uri("sqlite:///../mlflow.db")
mlflow.set_experiment("fraud-detection-paysim")

train = pd.read_parquet("../dataset/train.parquet")
test = pd.read_parquet("../dataset/test.parquet")

with open("../models/features.json") as f:
    FEATURES = json.load(f)
TARGET = "isFraud"

X_train, y_train = train[FEATURES], train[TARGET]
X_test, y_test = test[FEATURES], test[TARGET]

spw = (y_train == 0).sum() / (y_train == 1).sum()

params = {
    "n_estimators": 300,
    "max_depth": 6,
    "learning_rate": 0.1,
    "scale_pos_weight": float(spw),
}

with mlflow.start_run(run_name="xgb_baseline"):
    mlflow.log_params(params)
    mlflow.log_param("split_cutoff_step", 600)
    mlflow.log_param("n_features", len(FEATURES))

    model = XGBClassifier(
        **params,
        eval_metric="aucpr",
        tree_method="hist",
        n_jobs=-1,
        random_state=42,
    )
    model.fit(X_train, y_train)

    proba = model.predict_proba(X_test)[:, 1]
    pred = (proba >= 0.5).astype(int)

    mlflow.log_metric("pr_auc", average_precision_score(y_test, proba))
    mlflow.log_metric("precision", precision_score(y_test, pred))
    mlflow.log_metric("recall", recall_score(y_test, pred))
    mlflow.log_metric("f1", f1_score(y_test, pred))

    model.save_model("../models/xgb_baseline.json")
    mlflow.log_artifact("../models/xgb_baseline.json")
    mlflow.log_artifact("../models/features.json")

print("Run logged")


2026/11/04 10:52:16 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/11/04 10:52:16 INFO mlflow.store.db.utils: Updating database tables
2026/11/04 10:52:20 INFO mlflow.tracking.fluent: Experiment with name 'fraud-detection-paysim' does not exist. Creating a new experiment.


Run logged
